In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType, StringType, DoubleType , DateType

In [0]:
name_schema = StructType(fields=[StructField("forename",StringType(),True),
                                 StructField("surname",StringType(),True)])



In [0]:
drivers_schema = StructType(fields=[StructField("driverId",IntegerType(),True),
                                    StructField("driverRef",StringType(),True),
                                    StructField("number",IntegerType(),True),
                                    StructField("code",StringType(),
                                                True),
                                    StructField("name",name_schema),
                                    StructField("dob",DateType(),True),
                                    StructField("nationality",StringType(),True),
                                    StructField("url",StringType(),True)])

In [0]:
driver_df = spark.read.json("abfss://raw@databricksrg2026.dfs.core.windows.net/drivers.json")

In [0]:
display(driver_df)

Rename col and add ingestion

In [0]:
from pyspark.sql.functions import current_date , lit , concat , col

In [0]:
driver_rename_df = driver_df.withColumnRenamed("driverId","driver_id") \
                            .withColumnRenamed("driverRef","driver_ref") \
                            .withColumn("ingestion_date",current_date())\
                                .withColumn("name",concat(col("name.forename"),lit(" "),col("name.surname")))
                            

In [0]:
display(driver_rename_df)

In [0]:
driver_final_df = driver_rename_df.drop("url")

In [0]:
driver_final_df.write.mode("overwrite").parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/drivers")